# imports

In [ ]:
import sys
sys.path.append(r'\\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente\10_code\UTvsXCT-preprocessing')
from preprocess_tools import io, datasetmaker
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime


# Config

In [ ]:
# If True, samples whose VVF outputs (_vvf_maps.tif / _vvf_full_map.tif / _areafrac_map.tif)
# already exist on disk are skipped instead of recomputed.
SKIP_EXISTING_VVF = True

# Locate onlypores volumes

Discovers every `*_onlypores.tif` / `*_samplemask.tif` pair already produced by `onlypores_batch.ipynb`, under the same `probetas` root. This only depends on what has been processed so far, so it can be re-run later to pick up samples finished by a subsequent batch run.

In [ ]:
probetas_root = Path(r"/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas")

onlypores_paths = sorted(probetas_root.glob("*/onlypores files/*_onlypores.tif"))
print(f"Found {len(onlypores_paths)} processed onlypores volumes")
for p in onlypores_paths:
    print(" ", p)


# VVF computation

For each sample, this calls `datasetmaker.calculate_volume_vvf` — reusing `calculate_fraction_maps`'
front/middle/back depth-region split (also used by `datasetmaker_images.ipynb`) — applied directly
to the sample mask, with no UT pair involved. Unlike the UT-paired pipeline, the pore/mask volumes
are kept at their native XCT x-y resolution: no binning or cropping is applied, so the resulting
VVF maps have one value per XCT pixel. Whole-volume and per-region scalar VVF are computed on the
same (untouched) volumes.

Outputs saved alongside each source onlypores volume:
- `*_vvf_maps.tif`: front/middle/back per-pixel VVF maps, shape (3, x, y)
- `*_vvf_full_map.tif`: per-pixel VVF over the whole depth, shape (x, y)
- `*_areafrac_map.tif`: per-pixel area fraction
- `*_vvf_summary.csv`: that sample's row of the summary table (only this sample)

The combined table for all samples is still written to `onlypores_vvf_summary.csv` in the `probetas` root.

In [ ]:
MATERIAL_THRESHOLD = 0.8  # same default as datasetmaker.calculate_volume_vvf


def calculate_full_vvf_map(pores, mask, material_threshold=MATERIAL_THRESHOLD):
    """Per-pixel VVF over the whole depth (single region), shape (x, y) float32.

    Same conventions as the front/middle/back maps: -1 where there is no material
    or where the material content is below `material_threshold` of the maximum.
    """
    pore_sum = pores.sum(axis=0)
    mask_sum = mask.sum(axis=0).astype(np.float64)

    vvf_full_map = np.full(mask_sum.shape, -1, dtype=np.float32)
    valid = mask_sum > 0
    vvf_full_map[valid] = pore_sum[valid] / mask_sum[valid]

    full_material = mask_sum.max()
    if material_threshold is not None and full_material > 0:
        vvf_full_map[mask_sum / full_material < material_threshold] = -1

    return vvf_full_map


def sample_output_path(onlypores_path, suffix):
    return onlypores_path.with_name(onlypores_path.name.replace('_onlypores.tif', suffix))


def save_sample_summary(row, onlypores_path):
    """Write this sample's row (and only this one) next to its onlypores volume."""
    sample_summary_path = sample_output_path(onlypores_path, '_vvf_summary.csv')
    pd.DataFrame([row]).to_csv(sample_summary_path, index=False)
    return sample_summary_path


def compute_sample_vvf(onlypores_path):
    samplemask_path = sample_output_path(onlypores_path, '_samplemask.tif')
    sample_name = onlypores_path.parent.parent.name

    if not samplemask_path.exists():
        raise FileNotFoundError(f"Missing sample mask for {onlypores_path}: {samplemask_path}")

    pores = io.load_tif(onlypores_path) > 0
    mask = io.load_tif(samplemask_path) > 0

    volfrac_maps, areafrac_image, vvf_row = datasetmaker.calculate_volume_vvf(
        pores, mask, material_threshold=MATERIAL_THRESHOLD, metadata=True)
    vvf_full_map = calculate_full_vvf_map(pores, mask)

    vvf_maps_path = sample_output_path(onlypores_path, '_vvf_maps.tif')
    vvf_full_map_path = sample_output_path(onlypores_path, '_vvf_full_map.tif')
    areafrac_map_path = sample_output_path(onlypores_path, '_areafrac_map.tif')
    io.save_tif(vvf_maps_path, volfrac_maps.astype(np.float32))
    io.save_tif(vvf_full_map_path, vvf_full_map)
    io.save_tif(areafrac_map_path, areafrac_image.astype(np.float32))

    row = {
        'sample_name': sample_name,
        'onlypores_path': str(onlypores_path),
        'samplemask_path': str(samplemask_path),
        'vvf_maps_path': str(vvf_maps_path),
        'vvf_full_map_path': str(vvf_full_map_path),
        'areafrac_map_path': str(areafrac_map_path),
        **vvf_row,
    }

    return row


def vvf_outputs_exist(onlypores_path):
    return all(
        sample_output_path(onlypores_path, suffix).exists()
        for suffix in ('_vvf_maps.tif', '_vvf_full_map.tif', '_areafrac_map.tif')
    )


summary_path = probetas_root / 'onlypores_vvf_summary.csv'
existing_df = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()

rows = []
errors = []
skipped = 0

for onlypores_path in onlypores_paths:
    sample_name = onlypores_path.parent.parent.name

    if SKIP_EXISTING_VVF and vvf_outputs_exist(onlypores_path):
        existing_row = existing_df[existing_df['onlypores_path'] == str(onlypores_path)] if not existing_df.empty else existing_df
        if not existing_row.empty:
            print(f"Skipping {sample_name} (VVF already computed)")
            rows.append(existing_row.iloc[0].to_dict())
            save_sample_summary(rows[-1], onlypores_path)
            skipped += 1
            continue
        print(f"  outputs exist for {sample_name} but no matching row in {summary_path.name}; recomputing")

    print(f"Processing {sample_name}...")
    try:
        rows.append(compute_sample_vvf(onlypores_path))
        save_sample_summary(rows[-1], onlypores_path)
        print(f"  VVF total = {rows[-1]['vvf_total']*100:.4f}%")
    except Exception as e:
        print(f"  ERROR: {e}")
        errors.append((onlypores_path, str(e)))

vvf_df = pd.DataFrame(rows)
print(f"\n{len(rows)} samples in summary ({skipped} skipped, {len(rows) - skipped} computed, {len(errors)} errors)")
vvf_df

# Save results

In [ ]:
vvf_df.to_csv(summary_path, index=False)
print(f"Saved VVF summary to {summary_path} ({datetime.now().isoformat(timespec='seconds')})")